# 👑 KING2 AI — Fine-tuning Qwen2.5-3B-Instruct with Unsloth
إعادة تدريب نموذج KING2 على Qwen2.5-3B-Instruct باستخدام Unsloth + QLoRA


In [ ]:
# 1. تثبيت المكتبات
!pip install -qU unsloth
!pip install -qU --force-reinstall --no-deps unsloth
!pip install -qU huggingface_hub datasets

In [ ]:
# 2. تسجيل الدخول HuggingFace
from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(HF_TOKEN)
else:
    from huggingface_hub import login as hf_login
    hf_login()
from huggingface_hub import whoami
print(f'Logged in as: {whoami()["name"]}')

In [ ]:
# 3. رفع بيانات KING2 إلى HuggingFace Dataset
import json
from google.colab import files
print('ارفع king2_training_clean.json:')
uploaded = files.upload()
for fname in uploaded:
    with open(fname, 'r', encoding='utf-8') as f:
        training_data = json.load(f)
    print(f'Loaded {fname}: {len(training_data)} entries')
    break
from datasets import Dataset
ds = Dataset.from_list(training_data)
ds.push_to_hub('RASHID778/king2-training-data', private=False)
print('Dataset uploaded! https://huggingface.co/datasets/RASHID778/king2-training-data')

In [ ]:
# 4. تحميل النموذج + Unsloth
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
print(f'Model loaded! {model.num_parameters()/1e9:.2f}B params')

In [ ]:
# 5. إعداد LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42,
)
print(f'Trainable: {model.num_parameters(only_trainable=True)/1e6:.2f}M')

In [ ]:
# 6. تجهيز البيانات
from datasets import load_dataset
dataset = load_dataset('RASHID778/king2-training-data', split='train')
print(f'Loaded {len(dataset)} samples')

SYSTEM_PROMPT = 'أنت KING2، مساعد الذكاء الاصطناعي الملكي المتطور. أجب بالعربية الفصحى أولاً دائماً. كن مختصراً ومفيداً وواضحاً.'

def fmt(example):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for c in example['conversations']:
        r = 'user' if c['from'] == 'human' else 'assistant'
        msgs.append({'role': r, 'content': c['value']})
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(fmt)
print('Data ready!')

In [ ]:
# 7. التدريب
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=TrainingArguments(
        output_dir='./king2-qwen-lora',
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        logging_steps=5, save_steps=50, save_total_limit=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='cosine', max_steps=150,
        report_to='none',
    ),
    max_seq_length=2048, dataset_text_field='text',
)
trainer.train()
print('Training complete!')

In [ ]:
# 8. اختبار النموذج
FastLanguageModel.for_inference(model)
def ask(prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
for q in ['من أنت؟', 'ما هو الجمع؟', 'اشرح نظرية فيثاغورس', 'ما هو السلام الإيجابي؟']:
    print(f'Q: {q}\nA: {ask(q)}\n---')

In [ ]:
# 9. رفع النموذج إلى HuggingFace
from huggingface_hub import HfApi
model.save_pretrained('./king2-qwen-lora-final')
tokenizer.save_pretrained('./king2-qwen-lora-final')
HfApi().upload_folder(
    folder_path='./king2-qwen-lora-final',
    repo_id='RASHID778/king2-qwen2.5-3b',
    repo_type='model',
    commit_message='Update KING2 LoRA - cleaned data, 5 epochs, Unsloth',
)
print('Model uploaded! https://huggingface.co/RASHID778/king2-qwen2.5-3b')